## 1 Basics

In [20]:
import torch

In [21]:
torch.__version__

'2.8.0+cu128'

## 2 Understanding tensors

In [22]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
print(tensor1d.dtype)

torch.int64


## 3 Seeing models as computation graphs

## 4 Automatic differentiation made easy

In [23]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [24]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [25]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## 5 Implementing multilayer neural networks

In [26]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2rd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits


In [27]:
model = NeuralNetwork(50, 3)

In [28]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [29]:
num_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
print("Total number of trainale model parameters:", num_params)

Total number of trainale model parameters: 2213


In [30]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        [-0.0920, -0.0480,  0.0105,  ..., -0.0923,  0.1201,  0.0330],
        ...,
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509],
        [-0.1250,  0.0513,  0.0366,  ..., -0.1370,  0.1074, -0.0704]],
       requires_grad=True)


In [31]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [32]:
print(model.layers[0].bias)
print(model.layers[0].bias.shape)

Parameter containing:
tensor([-1.3122e-01, -1.0667e-02,  1.0314e-01, -1.8104e-02,  3.1523e-02,
        -1.3763e-01,  1.1643e-01,  9.1198e-02,  9.8382e-02,  7.3479e-02,
        -7.2337e-02, -1.1853e-01,  5.3997e-04,  7.5849e-02, -6.1513e-02,
         3.5053e-02, -1.1154e-02, -1.3147e-02,  3.6492e-02,  1.0322e-01,
         2.9582e-02,  1.0176e-02,  2.2896e-02,  2.6020e-02, -4.5835e-02,
        -1.6127e-02,  3.4467e-02, -1.1141e-01,  9.3448e-05, -1.3079e-01],
       requires_grad=True)
torch.Size([30])


In [33]:
torch.manual_seed(123)

model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [34]:
torch.manual_seed(123)

X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [35]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [36]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])


## 6 Setting up efficient data loaders

In [37]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0, 1])

In [42]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X 
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [ ]:
print(len(train_ds))
print(train_ds[1])

5
(tensor([-0.9000,  2.9000]), tensor(0))


In [65]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset = train_ds,
    batch_size = 2,
    shuffle = True,
    num_workers = 0,
    drop_last = True

)

In [66]:
test_ds = ToyDataset(X_test, y_test)

test_loader = DataLoader(
    dataset = test_ds,
    batch_size = 2,
    shuffle = False,
    num_workers = 0,
    drop_last = True
)

In [67]:
for idx, (x, y) in enumerate(train_loader):
    print(f'Batch {idx + 1}', x, y)

Batch 1 tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2 tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


## 7 A typical training loop

In [68]:
import torch.nn.functional as F 

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)


num_epochs = 3

for epoch in range(num_epochs):


    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)


        loss = F.cross_entropy(logits, labels)


        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


        ### LOADING
        print(f'Epoch {epoch+1:03d}/{num_epochs:03d}'
              f' | Batch {batch_idx:03d}/{len(train_loader):03d}'
              f' | Train/Val Loss: {loss:.2f}')
        
    model.eval()
    # Optional model evaluation


Epoch 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch 003/003 | Batch 001/002 | Train/Val Loss: 0.00
